# 2. Azure OpenAI generation

**Learning objective:** understand how a validated `LearningPathway` is produced
from a family intake plus supplied context, how the provider boundary is kept
small, and why generation was proved before retrieval existed.

**Where this fits:** this is the right-hand end of the pipeline.

```
intake -> [retrieval] -> context -> prompt -> Azure OpenAI -> LearningPathway
```

The bracketed step does not exist yet in this notebook. Context is assembled by
hand so that generation can be judged on its own.

## Why generation was proved first

A RAG system has two independent risks: retrieving the wrong evidence, and
writing badly from good evidence. If both are built at once, every bad result is
ambiguous.

So the first working slice of this project used a small set of hand-picked
passages as context. That answered one question in isolation: *given good
evidence, can the model produce something a family would actually want to read,
in a schema we can validate?* Only after that was answered did retrieval get
built.

In [ ]:
import inspect

from pydantic import ValidationError

from mosaic_pathway.generation import AzureOpenAIPathwayGenerator
from mosaic_pathway.models import (
    ChildProfile,
    FamilyIntake,
    LearningPathway,
)
from mosaic_pathway.prompts import SYSTEM_PROMPT, build_generation_prompt
from mosaic_pathway.settings import Settings, load_settings

print("generation modules imported")

In [ ]:
intake = FamilyIntake(
    children=[
        ChildProfile(
            label="older child",
            age=12,
            interests=["animals", "drawing"],
            learning_needs=["movement breaks"],
        )
    ],
    leaving_behind=["rigid daily schedules"],
    wants_to_preserve=["reading together after dinner"],
    wants_to_add=["more time outdoors"],
    family_values=["curiosity", "gentleness"],
    practical_constraints=["one working parent at home"],
    additional_context="We are in our first year of self-directed learning.",
)

manual_context = [
    {
        "source_id": "synthetic-guide-0007",
        "title": "Synthetic guide, chunk 7",
        "text": (
            "Families often start with one small repeated practice rather than a "
            "full timetable, and let the rhythm grow from what already works."
        ),
    },
    {
        "source_id": "synthetic-guide-0011",
        "title": "Synthetic guide, chunk 11",
        "text": (
            "An interest catalog is a simple shared list of what a child keeps "
            "returning to, used to choose the next resource together."
        ),
    },
]

print("context passages:", len(manual_context))
print("context keys:", sorted(manual_context[0]))

## System prompt versus user prompt

The split is deliberate and stable:

* the **system prompt** carries the role, the tone, and the rules that never
  change between families, including the rule that every recommendation must
  come from the supplied context
* the **user prompt** carries only the variable payload: this family's intake and
  this family's context

Keeping the rules out of the per-request payload means they cannot drift, and it
makes the request easier to reason about when a result looks wrong.

In [ ]:
system_lines = SYSTEM_PROMPT.splitlines()

print("system prompt lines:", len(system_lines))
print("system prompt characters:", len(SYSTEM_PROMPT))
print()
print("\n".join(system_lines[:6]))

In [ ]:
user_prompt = build_generation_prompt(intake, manual_context)

print("user prompt characters:", len(user_prompt))
print()
print(user_prompt[:300])
print("...")
print()
print("sections:", [line for line in user_prompt.splitlines() if line.isupper()])

## Structured output instead of free-form JSON

There are three common ways to get structured data out of a language model:

| Approach | What can go wrong |
| --- | --- |
| Ask for JSON in the prose | Fences, commentary, trailing text, invented fields |
| JSON mode | Valid JSON, but not necessarily the right shape |
| Structured output from a schema | The provider is constrained by the schema itself |

This project uses the third option. `client.beta.chat.completions.parse` is given
`response_format=LearningPathway`, so the same Pydantic model from notebook 1 is
both the schema sent to the provider and the type returned to the caller.

The contract still gets validated on the way back, because a provider can refuse
a request or return an unparsed message.

In [ ]:
incomplete_response = {
    "family_reflection": "A short reflection.",
    "starting_rhythm": [
        {
            "timing": "Most mornings",
            "practice": "Take a short walk.",
            "why_it_fits": "It adds outdoor time.",
        }
    ],
    "resources": [],
    "closing_note": "Go gently.",
}

try:
    LearningPathway.model_validate(incomplete_response)
except ValidationError as error:
    for item in error.errors():
        print(list(item["loc"]), "->", item["msg"])

Three separate failures are reported at once: too few rhythm practices, too few
resources, and a missing community suggestion. None of those would have been
caught by a `json.loads` call. This is why the schema is the contract rather than a
suggestion in the prompt.

## Authentication and configuration

There are no API keys in this project. `AzureOpenAIPathwayGenerator` builds a
bearer token provider from `DefaultAzureCredential`, so the caller's Entra ID
sign-in is what authorizes the request.

Only two values are configured, and both are read through a settings model rather
than scattered `os.environ` lookups.

In [ ]:
for name, field in Settings.model_fields.items():
    print(f"{name:<30} <- environment alias {field.alias}")

print()
print("settings loader:", load_settings.__name__)

The values themselves are not printed anywhere in this notebook. An endpoint and
a deployment name identify a specific Azure resource, so they stay in a local
`.env` file that is never committed.

One practical detail: `DefaultAzureCredential` reads the tenant from the process
environment, not from the settings model. If your sign-in spans several tenants,
export `AZURE_TENANT_ID` in the shell before launching Jupyter.

## The generator boundary

Everything provider-specific lives behind one class with one method. That method
signature is the only thing the rest of the system knows about Azure OpenAI.

In [ ]:
print("class :", AzureOpenAIPathwayGenerator.__name__)
print("init  :", inspect.signature(AzureOpenAIPathwayGenerator.__init__))
print("call  :", inspect.signature(AzureOpenAIPathwayGenerator.generate))

That narrow boundary buys three things:

* the RAG service in notebook 5 depends on a callable shape, not on a vendor SDK,
  so it can be exercised offline with a small fake
* swapping providers means writing one new class, not editing the pipeline
* failures are localized: a refusal or an unparsed message is raised as a
  `RuntimeError` at the boundary rather than leaking a provider object upward

## Optional live Azure OpenAI section

Everything above runs offline. The cells below are the only ones in this notebook
that contact Azure, and they are disabled by default.

To run them you need:

1. an Azure OpenAI deployment reachable from your account
2. `az login` completed in the shell that started Jupyter
3. `AZURE_TENANT_ID` exported in that same shell if your account spans tenants
4. `AZURE_OPENAI_BASE_URL` and `AZURE_OPENAI_CHAT_DEPLOYMENT` set in a local
   `.env` file

Set the flag below to `True` only when all four are true. The output prints the
family-facing pathway only.

In [ ]:
RUN_LIVE_AZURE = False

print("live Azure section enabled:", RUN_LIVE_AZURE)

In [ ]:
if RUN_LIVE_AZURE:
    generator = AzureOpenAIPathwayGenerator(load_settings())
    live_pathway = generator.generate(intake, manual_context)

    print(live_pathway.family_reflection)
    print()

    for practice in live_pathway.starting_rhythm:
        print(f"{practice.timing}: {practice.practice}")

    print()

    for resource in live_pathway.resources:
        print(f"{resource.title} [{resource.source_id}]")

    print()
    print(live_pathway.closing_note)
else:
    print("Skipped: set RUN_LIVE_AZURE to True to call Azure OpenAI.")

## The limitation this slice does not solve

A returned pathway carries a `source_id` on every resource. It is tempting to
read that as proof the recommendation is supported by that passage.

It is not. At this stage the id is only a string the model copied from the
context it was given. Nothing yet checks that the id was ever retrieved, and
nothing at all checks that the passage semantically supports the claim.

Notebook 5 adds the first of those checks. The second one is still, honestly, a
human job, which is why notebook 6 keeps a review rubric alongside the automated
suite.

## Key takeaways

* Generation was proved with hand-assembled context so that retrieval quality and
  writing quality could be judged separately.
* The system prompt holds invariant rules; the user prompt holds only this
  family's payload.
* Structured output makes the Pydantic schema the contract, and validation still
  runs on the way back.
* Authentication is Entra ID, not keys, and configuration is two values read
  through a settings model.
* All provider-specific code sits behind one class with one method.

## Next

Notebook 3 replaces the hand-assembled context with a real knowledge base built
from documents, which is where retrieval quality is actually decided.